# M04E: Capstone #1 — Research Assistant

Put it all together. Combine instructions, personas, few-shot, and quality scoring into one production-ready research assistant.

**Topics:**
- Multi-step research pipeline
- Question decomposition → Research → Synthesis → Validation
- All Module 4 techniques in one system

---

## 🔧 Step 1: Setup

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


# Truncation helper for long outputs
def truncate_response(text, max_length=1200):
    """Truncate text for readability."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + f"...\n\n💡 (Truncated from {len(text)} chars)"


print(f"✅ Setup complete: Using {MODEL}!")

---

## 🏗️ System Architecture

The research assistant combines all 4 techniques:
- **Instructions (M04A)** — Guide behavior at each step
- **Personas (M04B)** — Researcher role for expertise
- **Few-Shot (M04C)** — Examples for question decomposition
- **Quality Scoring (M04D)** — Validate output quality

**Pipeline:** Question → Decompose → Research → Synthesize → Validate



We'll build each phase step-by-step, with outputs flowing into the next phase. Then we'll package everything into a reusable `research_assistant()` function.

---

## 📝 Phase 1: Break Down Questions (Few-Shot)

Use few-shot examples to show how to decompose complex questions into answerable sub-questions.

In [ ]:
DECOMPOSITION_EXAMPLES = """Break complex questions into sub-questions:

Question: "How do smartphones affect daily productivity?"
Sub-questions:
[
  "What tasks do people use smartphones for during work?",
  "How does smartphone use impact focus and concentration?",
  "What are the time-saving benefits of smartphone apps?"
]

Question: "What makes a programming language popular?"
Sub-questions:
[
  "What factors influence language adoption?",
  "How do community and ecosystem affect popularity?",
  "What role does performance play in language choice?"
]
"""


# --------------------------------------------------------------
print("✅ Few-shot examples loaded")

### Test the Decomposition

In [ ]:
print("🔬 PHASE 1: QUESTION DECOMPOSITION")
print("="*60)

research_question = "What makes remote teams productive?"
print(f"Question: {research_question}\n")

prompt = f"{DECOMPOSITION_EXAMPLES}\nQuestion: \"{research_question}\"\nSub-questions:"

response = client.responses.create(
    model=MODEL,
    input=prompt,
    instructions="""Break into exactly 3 focused sub-questions.
Return as JSON array: ["question 1", "question 2", "question 3"]
No markdown fences. No other text."""
)

raw_output = response.output_text.strip()

# Clean up potential markdown code fences
cleaned_json = raw_output.replace("```json", "").replace("```", "").strip()

try:
    sub_questions_list = json.loads(cleaned_json)
    sub_questions_list = [q.strip() for q in sub_questions_list[:3]]
    
    if not sub_questions_list:
        sub_questions_list = [research_question]
except (json.JSONDecodeError, AttributeError):
    print(f"⚠️ Could not parse JSON, using fallback")
    sub_questions_list = [research_question]

print("Sub-questions:")
for i, q in enumerate(sub_questions_list, 1):
    print(f"  {i}. {q}")

print(f"\n✓ Parsed {len(sub_questions_list)} sub-questions")
print("="*60)

---

## 🔍 Phase 2: Research with Persona

Use researcher persona to answer each sub-question. We'll loop through the sub-questions from Phase 1.

In [ ]:
RESEARCHER_PERSONA = """You are an expert researcher who:
- Uses general knowledge to provide insights
- Considers multiple perspectives
- Identifies key patterns and trends
- Stays objective and balanced
- Does not invent sources or statistics
"""


# --------------------------------------------------------------
print("✅ Researcher persona loaded")

### Run the Research Step

In [ ]:
print("🔍 PHASE 2: RESEARCH WITH PERSONA")
print("="*60)

findings_list = []
for i, sub_q in enumerate(sub_questions_list, 1):
    print(f"Researching ({i}/{len(sub_questions_list)}): {sub_q}")
    
    response = client.responses.create(
        model=MODEL,
        input=sub_q,
        instructions=RESEARCHER_PERSONA + "\nProvide a concise answer (2-3 sentences)."
    )
    
    finding = response.output_text.strip()
    findings_list.append(finding)
    print(f"  → {finding}\n")

print(f"✓ Collected {len(findings_list)} findings")
print("="*60)

---

## 🎨 Phase 3: Synthesize Findings

Combine all research findings into a comprehensive, cohesive answer.

In [ ]:
print("🎨 PHASE 3: SYNTHESIZE FINDINGS")
print("="*60)

# Convert list to single numbered string with questions
findings_text = ""
for i, (q, f) in enumerate(zip(sub_questions_list, findings_list), 1):
    findings_text += f"{i}. Q: {q}\n   A: {f}\n\n"

synthesis_prompt = f"""Research question: {research_question}

Findings:
{findings_text}
Create comprehensive answer synthesizing all findings."""

response = client.responses.create(
    model=MODEL,
    input=synthesis_prompt,
    instructions=RESEARCHER_PERSONA + "\nSynthesize into cohesive answer. Be concise but complete. No headers."
)

answer = response.output_text.strip()
print(truncate_response(answer, max_length=600))
print("="*60)

---

## ⭐ Phase 4: Validate Quality

Score the synthesis to verify it meets standards before delivery.

In [ ]:
print("⭐ PHASE 4: VALIDATE QUALITY")
print("="*60)

validation_prompt = f"""Rate this research answer on a scale of 1-10:

Criteria:
- Completeness (addresses all aspects)
- Accuracy (consistent with general knowledge)
- Clarity (easy to understand)

Answer:
{answer}

Return ONLY a number from 1-10."""

response = client.responses.create(
    model=MODEL,
    input=validation_prompt,
    instructions="Return only a number from 1-10. Be concise."
)

raw_score = response.output_text.strip()

# Clean common variations like "8/10" or "Score: 8"
cleaned_score = raw_score.replace("/10", "").replace("Score:", "").strip()

try:
    score = float(cleaned_score)
    print(f"Quality Score: {score}/10")
    
    if score >= 8:
        print("✅ High quality - answer approved!")
    elif score >= 6:
        print("⚠️  Acceptable - consider refinement")
    else:
        print("❌ Low quality - regenerate recommended")
except ValueError:
    print(f"⚠️  Could not parse quality score: {raw_score}")

print("="*60)

### 🔑 Why This Works

Explicit criteria catch weak answers. A strict output format (number only) keeps parsing reliable.

---

## 🚀 Complete Research Assistant

Now let's combine all steps into one function.

In [ ]:
def research_assistant(question, validate=True, max_sub_questions=3):
    """Decompose → research → synthesize → validate"""

    print(f"🔬 Researching: {question}\n")

    # -------------------------------------------------------
    # Phase 1: Decompose question (Few-shot — M04C)
    # -------------------------------------------------------
    print("Phase 1: Breaking down question...")

    decompose_prompt = (
        f"{DECOMPOSITION_EXAMPLES}\n"
        f'Question: "{question}"\n'
        "Sub-questions:"
    )

    # Note: text parameter (M03A) could enforce JSON here,
    # but prompt-based approach keeps the pipeline readable
    decompose_response = client.responses.create(
        model=MODEL,
        input=decompose_prompt,
        instructions="""Break into exactly 3 sub-questions.
Return as JSON array: ["q1", "q2", "q3"]. No markdown fences. No other text."""
    )

    raw_output = decompose_response.output_text.strip()
    
    # Clean up potential markdown code fences
    cleaned_json = raw_output.replace("```json", "").replace("```", "").strip()
    
    try:
        sub_questions = json.loads(cleaned_json)
        sub_questions = [q.strip() for q in sub_questions[:max_sub_questions]]
        
        if not sub_questions:
            sub_questions = [question]
    except (json.JSONDecodeError, AttributeError):
        sub_questions = [question]

    print(f"  ✓ Generated {len(sub_questions)} sub-questions\n")

    # -------------------------------------------------------
    # Phase 2: Research sub-questions (Persona — M04B)
    # -------------------------------------------------------
    print("Phase 2: Researching sub-questions...")

    findings = []

    for i, sub_question in enumerate(sub_questions, 1):
        research_response = client.responses.create(
            model=MODEL,
            input=sub_question,
            instructions=RESEARCHER_PERSONA + "\nProvide a concise answer (2-3 sentences)."
        )

        findings.append(research_response.output_text.strip())
        print(f"  ✓ Researched {i}/{len(sub_questions)}")

    print()

    # -------------------------------------------------------
    # Phase 3: Synthesize (Instructions — M04A)
    # -------------------------------------------------------
    print("Phase 3: Synthesizing findings...")

    # Combine sub_questions and findings lists into a numbered string
    findings_text = ""
    for i, (q, f) in enumerate(zip(sub_questions, findings), 1):
        findings_text += f"{i}. Q: {q}\n   A: {f}\n\n"

    synthesis_prompt = f"""Research question: {question}

Findings:
{findings_text}
Create comprehensive answer synthesizing all findings."""

    synthesis_response = client.responses.create(
        model=MODEL,
        input=synthesis_prompt,
        instructions=RESEARCHER_PERSONA + "\nSynthesize into cohesive answer. Be concise but complete. No headers."
    )

    answer = synthesis_response.output_text.strip()
    print("  ✓ Synthesis complete\n")

    # -------------------------------------------------------
    # Phase 4: Validate (Quality Scoring — M04D)
    # -------------------------------------------------------
    quality_score = None

    if validate:
        print("Phase 4: Validating quality...")

        validation_prompt = f"""Rate this research answer on a scale of 1-10:

Criteria:
- Completeness (addresses all aspects)
- Accuracy (consistent with general knowledge)
- Clarity (easy to understand)

Answer:
{answer}

Return ONLY a number from 1-10."""

        score_response = client.responses.create(
            model=MODEL,
            input=validation_prompt,
            instructions="Return only a number from 1-10. Be concise."
        )

        raw_score = score_response.output_text.strip()
        
        # Clean common variations like "8/10" or "Score: 8"
        cleaned_score = raw_score.replace("/10", "").replace("Score:", "").strip()
        
        try:
            quality_score = float(cleaned_score)
            print(f"  ✓ Quality: {quality_score}/10\n")
        except ValueError:
            print(f"  ⚠️  Could not parse quality score: {raw_score}\n")

    return {
        'question': question,
        'sub_questions': sub_questions,
        'findings': findings,
        'answer': answer,
        'quality_score': quality_score
    }


# --------------------------------------------------------------
print("✅ Research assistant ready!")

---

## 🎬 Demo: Full Pipeline

In [ ]:
print("🚀 FULL RESEARCH ASSISTANT DEMO")
print("="*60)
print()

result = research_assistant("What makes remote teams productive?")

# Display results
print("="*60)
print("📊 RESEARCH RESULTS")
print("="*60)

print(f"\n❓ Question: {result['question']}")
print(f"\n📝 Sub-questions: {len(result['sub_questions'])}")
print(f"🔍 Findings: {len(result['findings'])}")

if result['quality_score'] is not None:
    print(f"\n⭐ Quality: {result['quality_score']:.1f}/10")

print("\n✅ FINAL ANSWER:")
print(truncate_response(result['answer'], max_length=400))
print("="*60)

### 💡 What We Built

A production system combining all Module 4 techniques:
- **M04A:** Instructions guiding each step
- **M04B:** Researcher persona throughout
- **M04C:** Few-shot for question decomposition
- **M04D:** Quality validation with scoring

---

### 💪 Your Turn: Extend the System

Add iterative refinement — if the quality score is below 7, automatically ask the model to improve the synthesis until it passes.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Extend the Research Assistant
# --------------------------------------------------------------
# Objective: Add quality checking and iterative refinement.
# Steps 1-3 are provided from research_assistant above.
# Your job: build the refinement loop in Step 4.
# ⚠️ Always use max_refinements (e.g., 2) to prevent infinite API loops!

def enhanced_research_assistant(question, max_sub_questions=3, max_refinements=2):
    """Research assistant with iterative refinement."""
    
    # Step 1: Decompose
    decompose_response = client.responses.create(
        model=MODEL,
        input=f"Break this into {max_sub_questions} sub-questions:\n{question}",
        instructions="Return only numbered sub-questions."
    )
    questions = [q.strip() for q in decompose_response.output_text.strip().split("\n") if q.strip()]
    
    # Step 2: Research
    findings = []
    for q in questions:
        response = client.responses.create(
            model=MODEL,
            input=q,
            instructions="Give a concise, factual answer."
        )
        findings.append(response.output_text.strip())
    
    # Combine questions and findings lists into a numbered string
    findings_text = ""
    for i, (q, f) in enumerate(zip(questions, findings), 1):
        findings_text += f"{i}. Q: {q}\n   A: {f}\n\n"
    
    # Step 3: Synthesize
    synthesis_response = client.responses.create(
        model=MODEL,
        input=f"Question: {question}\n\nResearch:\n{findings_text}",
        instructions="Synthesize a comprehensive answer from the research."
    )
    current_answer = synthesis_response.output_text.strip()
    
    # Step 4 (NEW): Iterative refinement — YOUR CODE HERE
    refinements = 0
    while refinements < max_refinements:
        # TODO: Score current_answer (1-10) using quality scoring from M04D
        # TODO: If score >= 7, break
        # TODO: Otherwise, ask the model to improve current_answer
        refinements += 1
    
    return current_answer


# --------------------------------------------------------------
# TODO: Test your enhanced assistant
# result = enhanced_research_assistant("What makes remote teams productive?")

print("💡 Build your enhanced research assistant!")

---

## 🎯 Key Takeaways

**🔗 Combine Techniques Into Pipelines:**
- Single techniques solve single problems — pipelines solve real tasks
- Break complex tasks into steps, apply the best technique per step
- Few-shot for decomposition, personas for expertise, scoring for validation

**✅ Validate Critical Outputs:**
- Quality scoring catches errors that single responses miss
- Add validation steps for any output that drives decisions

**The Flow:** Decompose with few-shot → Research with persona → Synthesize with instructions → Validate with scoring

---

### 🎉 Module 4 Complete!

You've built systems using instructions, personas, few-shot examples, and quality scoring — individually and combined.

---

### 📍 Next Step

**M05A: Context & Token Management** — Count tokens, track costs, and manage context windows to keep your prompts production-ready.

---

## 🔧 Troubleshooting

**Research quality inconsistent?**
- Add more few-shot examples
- Strengthen persona description
- Use clearer research questions

**Sub-questions not helpful?**
- Improve few-shot examples
- Add specificity to instructions
- Show edge cases in examples

**Synthesis too shallow?**
- Include all findings in prompt
- Strengthen synthesis instructions
- Ask for connections between findings

**Quality scores unreliable?**
- Make scoring criteria specific
- Enforce "number only" output format

**Too expensive?**
- Reduce sub-questions (2 instead of 3)
- Skip validation for simple queries

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---